# Deception direction from MASK — Qwen3-8B

Runs the generation and activation half of `deception_mask/mask_deception.py` (see `deception_mask/DESIGN.md`):

1. **gen**: belief elicitations, 4 pressure samples per MASK item, the instructed / control / confession sets (about 4,500 generations)
2. **acts**: mean residual-stream activation over each response, every layer, plus entropy and length
3. **export_judge**: judge packets for blind Claude judges (no labels inside)

Judging and the analysis run afterwards on the Mac.

**Runtime → Change runtime type → A100 GPU** (L4 also works, about 3× slower). Keep this tab open (Colab Pro stops
idle background runs). Everything is saved to Google Drive as it goes, so if the runtime disconnects, just
**Run all** again and it resumes.

Expected: about 45 min on an A100, about 2 h on an L4.

In [ ]:
import subprocess, torch
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> A100 (or L4)'
mem = torch.cuda.get_device_properties(0).total_memory / 2**30
assert torch.cuda.is_bf16_supported() and mem > 22, f'Need an A100 / L4 (bf16, >22 GB); got {torch.cuda.get_device_name(0)} {mem:.0f} GB'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/deception_mask/out_colab'
import os; os.makedirs(OUT, exist_ok=True); print('outputs ->', OUT, os.listdir(OUT))

In [ ]:
# code + data, embedded (built by deception_mask/colab/build.py)
import base64, io, zipfile
BLOB = '...'
zipfile.ZipFile(io.BytesIO(base64.b64decode(BLOB))).extractall('/content/repo')
print(sorted(zipfile.ZipFile(io.BytesIO(base64.b64decode(BLOB))).namelist()))

In [ ]:
!pip install -q -U "transformers>=4.56" accelerate

In [ ]:
import subprocess, sys, time
# A100 (40/80 GB) fits 48 sequences at once; an L4 (24 GB) holds the 16 GB model plus ~16 sequences of KV cache
BS = ['--gen_bs', '48', '--act_bs', '4'] if mem > 30 else ['--gen_bs', '16', '--act_bs', '2']
def stage(name):
    cmd = [sys.executable, '-u', '/content/repo/deception_mask/mask_deception.py', '--preset', 'colab', '--out', OUT, '--stage', name] + BS
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    n = 0
    log = open(f'{OUT}/run.log', 'a')   # full log on Drive, so progress can be followed from the synced folder
    log.write(f'=== {time.strftime("%Y-%m-%d %H:%M:%S")} UTC stage {name} on {torch.cuda.get_device_name(0)}\n'); log.flush()
    for line in p.stdout:   # progress lines come every batch; show every 25th
        log.write(f'{time.strftime("%H:%M:%S")} {line}'); log.flush()
        if line.startswith(('  generated', '  activations')):
            n += 1
            if n % 25: continue
        print(line, end='', flush=True)
    rc = p.wait(); log.write(f'=== stage {name} exit {rc}\n'); log.close()
    assert rc == 0, f'stage {name} failed'
stage('gen')      # resumable: answers are saved to Drive every 20 batches

In [ ]:
stage('acts')     # writes acts.pt (~1 GB) to Drive

In [ ]:
stage('export_judge')
import shutil
z = shutil.make_archive('/content/deception_colab_results', 'zip', OUT, '.')
shutil.copy(z, '/content/drive/MyDrive/deception_mask/deception_colab_results.zip')
print('DONE ->', '/content/drive/MyDrive/deception_mask/deception_colab_results.zip', os.path.getsize(z) // 2**20, 'MB')

## Done

Download **`MyDrive/deception_mask/deception_colab_results.zip`** from Google Drive and put it in
`personality-axes/deception_mask/` on the Mac. Claude then judges the packets blind and runs the analysis.